# Validate Code Eval Pipeline (MBPP + LiveCodeBench)\n\nTest the full pipeline: dataset loading → generation → code execution → correctness check.\nUsing **Qwen3-0.6B** (base model, no SoRL) as a quick sanity check.

In [2]:
import sys, os

# Ensure project root is on sys.path (handles both local and /workspace/mod_gpt)
_nb_dir = os.getcwd()
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)
# Also try common remote path
if "/workspace/mod_gpt" not in sys.path:
    sys.path.insert(0, "/workspace/mod_gpt")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from data.pt_dataset import (
    get_dataset, MBPPDataset, LiveCodeBenchDataset,
    sandbox_execute, check_code_correctness,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen3-0.6B"
print(f"Device: {DEVICE}")
print(f"CWD: {_nb_dir}")

ModuleNotFoundError: No module named 'data.pt_dataset'

## 1. Load Model & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16
).to(DEVICE).eval()
print(f"Model loaded: {MODEL_NAME} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

## 2. Load Datasets & Inspect Samples

In [ ]:
# Load MBPP
mbpp_ds = get_dataset("mbpp", split="test", tokenizer=tokenizer, max_length=1024)
print(f"MBPP test: {len(mbpp_ds)} problems")

# Inspect a sample
ex = mbpp_ds.dataset[0]
print(f"\n--- MBPP Sample 0 ---")
print(f"Prompt: {ex['prompt'][:200]}")
print(f"Code: {ex['code'][:200]}")
tests = mbpp_ds.get_test_cases(0)
print(f"Tests ({len(tests)}): {tests[:3]}")

In [ ]:
# Load LiveCodeBench
lcb_ds = get_dataset("livecodebench", split="test", tokenizer=tokenizer, max_length=1024)
print(f"LiveCodeBench: {len(lcb_ds)} problems")

# Inspect a sample — print column names first to verify schema
ex = lcb_ds.dataset[0]
print(f"\nColumn names: {list(ex.keys())}")
print(f"\n--- LiveCodeBench Sample 0 ---")
question = ex.get("question_content", ex.get("question", ""))
print(f"Question: {str(question)[:300]}...")
print(f"Difficulty: {ex.get('difficulty', 'unknown')}")
print(f"Starter code: {ex.get('starter_code', 'none')[:100]}")
tests = lcb_ds.get_test_cases(0)
print(f"Tests ({len(tests)}): {type(tests[0]) if tests else 'none'}")
if tests:
    print(f"  preamble: {tests[0]['preamble'][:100]}...")
    print(f"  check: {tests[0]['check'][:100]}...")

## 3. Sanity Check: Execute Ground-Truth Solutions\n\nBefore testing model generations, verify that the sandbox + test cases work on the **reference solutions**.

In [ ]:
import json

# --- MBPP: test ground-truth solutions against their own test cases ---
print("=== MBPP: Ground-truth solution check ===")
n_check = 5
for i in range(n_check):
    ex = mbpp_ds.dataset[i]
    code = ex["code"]
    tests = mbpp_ds.get_test_cases(i)
    result = check_code_correctness(code, tests, timeout=10)
    status = "✅ PASS" if result["passed"] else f"❌ FAIL ({result['errors'][:1]})"
    print(f"  Problem {i}: {status}  ({result['num_passed']}/{result['num_total']} tests)")

# --- LiveCodeBench: no ground-truth solutions, just verify test case parsing ---
print("\n=== LiveCodeBench: Test case parsing check ===")
lcb_with_tests = 0
for i in range(min(50, len(lcb_ds))):
    tests = lcb_ds.get_test_cases(i)
    if tests:
        lcb_with_tests += 1
        if lcb_with_tests <= 3:
            print(f"  Problem {i}: {len(tests)} test cases found")
print(f"  {lcb_with_tests}/50 problems have parseable test cases")

## 4. Generate Code with Qwen3-0.6B\n\nTest the model's ability to generate code for a few MBPP and APPS problems.

In [ ]:
@torch.no_grad()
def generate_code(model, tokenizer, prompt_text, max_new_tokens=256):
    """Generate code from a prompt using the base HF model."""
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.0,   # greedy
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Decode only the new tokens
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# --- MBPP: generate + execute ---
print("=== MBPP: Model Generation + Execution ===\n")
N_MBPP = 5
for i in range(N_MBPP):
    ex = mbpp_ds.dataset[i]
    prompt = f"# Task: {ex['prompt'].strip()}\n# Solution:\n"
    generated = generate_code(model, tokenizer, prompt, max_new_tokens=256)
    tests = mbpp_ds.get_test_cases(i)
    result = check_code_correctness(generated, tests, timeout=10)
    status = "✅" if result["passed"] else "❌"
    
    print(f"--- Problem {i} {status} ({result['num_passed']}/{result['num_total']}) ---")
    print(f"Task: {ex['prompt'][:120]}...")
    print(f"Generated:\n{generated[:300]}")
    if result["errors"]:
        print(f"Errors: {result['errors'][:1]}")
    print()

In [ ]:
# --- LiveCodeBench: generate + execute ---
print("=== LiveCodeBench: Model Generation + Execution ===\n")
N_LCB = 5
lcb_tested = 0
for i in range(min(100, len(lcb_ds))):
    tests = lcb_ds.get_test_cases(i)
    if not tests:
        continue
    
    ex = lcb_ds.dataset[i]
    question = ex.get("question_content", ex.get("question", "")).strip()
    starter = ex.get("starter_code", "").strip()
    if starter:
        prompt = f"# Problem:\n{question}\n\n{starter}\n# Solution:\n"
    else:
        prompt = f"# Problem:\n{question}\n\n# Solution:\n"
    
    generated = generate_code(model, tokenizer, prompt, max_new_tokens=512)
    result = check_code_correctness(generated, tests, timeout=10)
    status = "✅" if result["passed"] else "❌"
    
    print(f"--- Problem {i} [{ex.get('difficulty', '?')}] {status} ({result['num_passed']}/{result['num_total']}) ---")
    print(f"Question: {question[:150]}...")
    print(f"Generated:\n{generated[:300]}")
    if result["errors"]:
        print(f"Errors: {result['errors'][:1]}")
    print()
    
    lcb_tested += 1
    if lcb_tested >= N_LCB:
        break

## 5. Batch Eval: Pass@1 on N samples\n\nRun a proper pass@1 evaluation loop on a small subset to validate the full pipeline.

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def eval_pass_at_1(model, tokenizer, dataset, n_samples=20, max_new_tokens=256, timeout=10):
    """Compute pass@1 on a code dataset with execution-based checking."""
    passed = 0
    tested = 0
    skipped = 0
    
    for i in tqdm(range(min(n_samples * 3, len(dataset))), desc="Evaluating"):
        # Get test cases; skip problems without tests
        tests = dataset.get_test_cases(i)
        if not tests:
            skipped += 1
            continue
        if tested >= n_samples:
            break
        
        # Build prompt and generate
        ex = dataset.dataset[i]
        if isinstance(dataset, MBPPDataset):
            prompt = f"# Task: {ex['prompt'].strip()}\n# Solution:\n"
        else:  # LiveCodeBench
            question = ex.get("question_content", ex.get("question", "")).strip()
            starter = ex.get("starter_code", "").strip()
            if starter:
                prompt = f"# Problem:\n{question}\n\n{starter}\n# Solution:\n"
            else:
                prompt = f"# Problem:\n{question}\n\n# Solution:\n"
        
        generated = generate_code(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        result = check_code_correctness(generated, tests, timeout=timeout)
        
        if result["passed"]:
            passed += 1
        tested += 1
    
    acc = passed / max(tested, 1)
    print(f"\npass@1 = {passed}/{tested} = {acc:.1%}  (skipped {skipped} problems without tests)")
    return {"pass_at_1": acc, "passed": passed, "tested": tested, "skipped": skipped}


# --- MBPP pass@1 ---
print("=" * 50)
print("MBPP pass@1 (20 samples)")
print("=" * 50)
mbpp_result = eval_pass_at_1(model, tokenizer, mbpp_ds, n_samples=20, max_new_tokens=256)

print()

# --- LiveCodeBench pass@1 ---
print("=" * 50)
print("LiveCodeBench pass@1 (20 samples)")
print("=" * 50)
lcb_result = eval_pass_at_1(model, tokenizer, lcb_ds, n_samples=20, max_new_tokens=512)

## 6. Summary\n\nKey things to verify:\n1. **Ground-truth check (§3)**: MBPP reference solutions should pass their own tests (✅ = sandbox works)\n2. **Test case parsing (§3)**: LiveCodeBench problems have parseable test cases\n3. **Generation (§4)**: Model produces syntactically valid Python (even if incorrect)\n4. **Execution (§4)**: Tests correctly catch wrong outputs and pass correct ones\n5. **Batch eval (§5)**: pass@1 numbers are reasonable for a 0.6B base model (expect ~10-30% MBPP, ~1-5% LiveCodeBench)

# TODO: delete this duplicate cell